# Neural Network from Scratch — Gradient Derivations

This notebook documents the forward and backward equations for the NumPy-only MNIST MLP.


## Forward Pass

For mini-batch $X \in \mathbb{R}^{m \times 784}$:

$$Z^{[1]} = XW^{[1]} + b^{[1]}$$
$$R^{[1]} = \max(0, Z^{[1]})$$
$$\mu_B^{[1]} = \frac{1}{m}\sum_i R_i^{[1]}, \quad \sigma_B^{2[1]} = \frac{1}{m}\sum_i (R_i^{[1]}-\mu_B^{[1]})^2$$
$$\hat{R}^{[1]} = \frac{R^{[1]}-\mu_B^{[1]}}{\sqrt{\sigma_B^{2[1]}+\epsilon}}$$
$$B^{[1]} = \gamma^{[1]}\hat{R}^{[1]} + \beta^{[1]}$$
$$A^{[1]} = B^{[1]} \odot \frac{M^{[1]}}{1-p_1}$$

The same ReLU, BatchNorm, and Dropout pattern is repeated for layer 2, then:

$$Z^{[3]} = A^{[2]}W^{[3]} + b^{[3]}$$
$$\hat{Y}_{ij} = \frac{e^{Z^{[3]}_{ij} - \max_k Z^{[3]}_{ik}}}{\sum_k e^{Z^{[3]}_{ik} - \max_r Z^{[3]}_{ir}}}$$
$$L = -\frac{1}{m}\sum_i\sum_j Y_{ij}\log(\hat{Y}_{ij})$$


## Softmax + Cross-Entropy

Combining softmax with cross-entropy simplifies the output gradient:

$$\delta^{[3]} = \frac{\partial L}{\partial Z^{[3]}} = \frac{\hat{Y} - Y}{m}$$

Therefore:

$$\frac{\partial L}{\partial W^{[3]}} = A^{[2]T}\delta^{[3]}$$
$$\frac{\partial L}{\partial b^{[3]}} = \sum_i \delta_i^{[3]}$$
$$\frac{\partial L}{\partial A^{[2]}} = \delta^{[3]}W^{[3]T}$$


## Dropout Backward

For inverted dropout $A = B \odot M/(1-p)$:

$$\frac{\partial L}{\partial B} = \frac{\partial L}{\partial A} \odot \frac{M}{1-p}$$


## BatchNorm Backward

Let $G = \partial L / \partial B$. The scale and shift gradients are:

$$\frac{\partial L}{\partial \gamma} = \sum_i G_i\hat{R}_i, \qquad \frac{\partial L}{\partial \beta} = \sum_i G_i$$

The normalized input gradient starts with:

$$\frac{\partial L}{\partial \hat{R}} = G \odot \gamma$$

Variance and mean gradients:

$$\frac{\partial L}{\partial \sigma_B^2} = \sum_i \frac{\partial L}{\partial \hat{R}_i}(R_i-\mu_B)(-\frac{1}{2})(\sigma_B^2+\epsilon)^{-3/2}$$
$$\frac{\partial L}{\partial \mu_B} = \sum_i -\frac{\partial L}{\partial \hat{R}_i}(\sigma_B^2+\epsilon)^{-1/2} + \frac{\partial L}{\partial \sigma_B^2}\frac{1}{m}\sum_i -2(R_i-\mu_B)$$

Input gradient:

$$\frac{\partial L}{\partial R_i} = \frac{\partial L}{\partial \hat{R}_i}(\sigma_B^2+\epsilon)^{-1/2} + \frac{\partial L}{\partial \sigma_B^2}\frac{2(R_i-\mu_B)}{m} + \frac{1}{m}\frac{\partial L}{\partial \mu_B}$$


## ReLU and Dense Backward

ReLU:

$$\delta^{[l]} = \frac{\partial L}{\partial R^{[l]}} \odot \mathbb{1}[Z^{[l]} > 0]$$

Dense layer $Z^{[l]} = A^{[l-1]}W^{[l]} + b^{[l]}$:

$$\frac{\partial L}{\partial W^{[l]}} = A^{[l-1]T}\delta^{[l]}$$
$$\frac{\partial L}{\partial b^{[l]}} = \sum_i \delta_i^{[l]}$$
$$\frac{\partial L}{\partial A^{[l-1]}} = \delta^{[l]}W^{[l]T}$$


## Adam

$$m_t = \beta_1m_{t-1} + (1-\beta_1)g_t$$
$$v_t = \beta_2v_{t-1} + (1-\beta_2)g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
$$\theta_t = \theta_{t-1} - \alpha\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}$$
